# Argus VLM Optimization — Notebook 05: Spatial Redundancy Reduction

**Goal:** Evaluate grid-based change detection and bounding box cropping to feed only the active dynamic region to the VLM, cutting visual token computation.


In [ ]:
# Cell 1: Install Dependencies
!pip install -q torch transformers accelerate pillow pyyaml pandas matplotlib seaborn opencv-python-headless rouge-score


In [ ]:
# Cell 2: Imports & Environment Check
import os
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
from PIL import Image, ImageDraw
from src.frame_optimization.spatial_redundancy import PatchChangeDetector, crop_changed_region
from src.visualization.plots import plot_token_reduction


In [ ]:
# Cell 3: Configuration
patch_sizes = [16, 32, 64]
change_thresholds = [0.05, 0.10, 0.15]
padding_values = [16, 32]


In [ ]:
# Cell 4: Model Loading (Optional for spatial crop geometry)
# Token estimation can be verified via resolution or actual VLM wrapper


In [ ]:
# Cell 5: Frame Pair Preparation
prev_frame = Image.new("RGB", (640, 480), color=(100, 100, 100))
curr_frame = Image.new("RGB", (640, 480), color=(100, 100, 100))
d = ImageDraw.Draw(curr_frame)
d.rectangle([200, 150, 280, 230], fill=(255, 255, 0)) # localized change


In [ ]:
# Cell 6: Spatial Sweep
records = []
output_csv = repo_root / "results" / "static_frames" / "spatial_results.csv"

for pz in patch_sizes:
    for ct in change_thresholds:
        for pad in padding_values:
            detector = PatchChangeDetector(patch_size=pz, change_threshold=ct)
            det_res = detector.detect(prev_frame, curr_frame)
            cropped, actual_ratio = crop_changed_region(curr_frame, det_res, padding=pad)
            records.append({
                "patch_size": pz,
                "change_threshold": ct,
                "padding": pad,
                "has_changed_region": det_res.has_changed_region,
                "crop_w": cropped.size[0],
                "crop_h": cropped.size[1],
                "crop_area_ratio": actual_ratio,
                "estimated_token_savings_pct": (1.0 - actual_ratio) * 100.0
            })

df = pd.DataFrame(records)
df.to_csv(output_csv, index=False)
print(df.head())


In [ ]:
# Cell 7: Plot Results
plot_token_reduction(df, output_path=str(repo_root / "results" / "figures" / "spatial_token_reduction.png"))
